# BTCUSD Multi-Timeframe Transformer Training

Trains independent per-TF transformers for all supported resolutions (5m, 15m, 30m, 1h, 2h).
Shared modules: `feature_store/transformer_btcusd/`.

Each TF exports to its own subdirectory under `export_dir`:

```
export/
├── JackSparrow_Transformer_BTCUSD_5m/
│   ├── metadata_transformer.json
│   ├── btcusd_5m_transformer.onnx
│   └── feature_config.json
├── JackSparrow_Transformer_BTCUSD_15m/
└── ...
```

Copy each subdirectory into `agent/model_storage/` after training.

In [ ]:
!pip install -q torch pandas numpy scikit-learn pyarrow onnx onnxruntime requests

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
for candidate in [ROOT, ROOT / "JackSparrow"]:
    if (candidate / "feature_store").is_dir():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not find feature_store/")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f"Using repo root: {ROOT}")

In [ ]:
from pathlib import Path

from feature_store.transformer_btcusd.contract import SUPPORTED_RESOLUTIONS

resolutions = list(SUPPORTED_RESOLUTIONS)  # or subset: ["15m", "1h"]
export_dir = Path("/content/export")  # Path("export") for local
epochs = None  # e.g. 5 for smoke test
history_days = None
refresh_data = False
continue_on_error = True  # finish remaining TFs if one fails quality gate

In [ ]:
from scripts.colab.train_transformer_resolution import run_all_training

results = run_all_training(
    resolutions=resolutions,
    export_dir=export_dir,
    epochs=epochs,
    history_days=history_days,
    refresh_data=refresh_data,
    continue_on_error=continue_on_error,
)

for result in results:
    print(result)

In [ ]:
import shutil
from pathlib import Path

zip_path = shutil.make_archive("/content/transformer_exports", "zip", export_dir)
print(f"Download: {zip_path}")